### 추천 : 컨텐츠 기반 추천 개발 파이프라인

- 이 노트북은 **`컨텐츠 기반 추천` 개발 파이프라인 예제 실습**을 수행하는 노트북입니다.

##### 컨텐츠 기반 추천 개발 파이프라인 예제
   1.  상품 별 감정 분석 데이터 로드 및 전처리 (Cell 4개)
   2.  TF-IDF 벡터화 기반 콘텐츠 추천 시스템 (Cell 2개)
   3.  LSA 차원 축소를 통한 추천 시스템 (Cell 2개)
   4.  PCA 주성분 분석 기반 추천 시스템 (Cell 2개)
   5.  K-means 클러스터링을 통한 특징 공간 품질 평가 (Cell 2개)
   6.  종합 성능 비교 분석 및 최적 방법론 도출 (Cell 3개)


### 01: 상품 별 감정 분석 데이터 로드 및 전처리
JSON 형태의 상품 별 감정 분석 데이터를 pandas DataFrame으로 변환하고 추천 시스템에 적합한 형태로 전처리합니다. 

상품 별 리뷰를 통합하고 속성별 감정 점수를 계산하여 추천 알고리즘의 입력 데이터로 준비합니다.

* 필수 라이브러리(pandas, numpy, scikit-learn 등) 임포트 및 기본 설정
* JSON 파일을 재귀적으로 탐색하여 모든 리뷰 데이터를 하나의 DataFrame으로 결합
* 결측값 처리, 데이터 타입 변환 및 속성별 감정 점수 추출하여 AspectScores 컬럼 생성
* 제품별 리뷰 텍스트 통합 및 메타데이터(도메인, 카테고리, 감정점수) 집계
* 전처리 완료된 데이터의 통계 정보 출력 및 추천 시스템용 데이터 구조 생성

In [ ]:
# ============================================ 
# Cell 1: 라이브러리 임포트 및 기본 설정
# ============================================ 

import pandas as pd              
import numpy as np               
import json                      
import os                        
from glob import glob            
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer  
from sklearn.metrics.pairwise import cosine_similarity                        
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import KMeans                                            
from sklearn.preprocessing import StandardScaler                              
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score  

import time                      
import warnings                  
warnings.filterwarnings('ignore')  

print("="*60)
print("1. 데이터 로드 및 전처리")
print("="*60)


1. 데이터 로드 및 전처리


In [17]:
# ============================================ 
# Cell 2: 데이터 로드 함수 정의
# ============================================ 

def load_json_data(file_path):
    """
    단일 JSON 파일을 로드하여 pandas DataFrame으로 변환하는 함수
    
    Parameters:
    -----------
    file_path : str
        로드할 JSON 파일의 절대 경로 또는 상대 경로
        
    Returns:
    --------
    pd.DataFrame
        JSON 데이터가 pandas DataFrame 형태로 변환된 객체
        각 행은 하나의 리뷰, 각 열은 리뷰의 속성(제품명, 텍스트, 감정점수 등)
        
    Raises:
    -------
    FileNotFoundError : 파일이 존재하지 않을 때
    json.JSONDecodeError : JSON 형식이 올바르지 않을 때
    """
    with open(file_path, 'r', encoding='utf-8') as f:  
        data = json.load(f)  
    return pd.DataFrame(data)  

def load_all_json_files(base_path):
    """
    지정된 기본 경로에서 모든 JSON 파일을 재귀적으로 찾아 하나의 DataFrame으로 결합
    
    이 함수는 대용량 데이터셋에서 여러 폴더에 분산된 JSON 파일들을 
    자동으로 찾아서 하나의 통합된 데이터셋으로 만드는 역할을 합니다.
    
    Parameters:
    -----------
    base_path : str
        JSON 파일들이 저장된 최상위 디렉토리 경로
        일반적으로 'Training/02.라벨링데이터' 형태의 구조를 가정
        
    Returns:
    --------
    pd.DataFrame
        모든 JSON 파일의 데이터가 행 방향으로 결합된 DataFrame
        각 파일의 카테고리 정보가 'CategoryPath' 컬럼으로 추가됨
        
    Note:
    -----
    - 파일 로드 중 오류가 발생한 파일은 건너뛰고 오류 메시지 출력
    - 메모리 사용량 최적화를 위해 리스트에 임시 저장 후 한 번에 결합
    """
    all_data = []  
    
    train_path = os.path.join(base_path, 'Training', '02.라벨링데이터')
    
    json_files = glob(os.path.join(train_path, '**', '*.json'), recursive=True)
    
    print(f"찾은 JSON 파일 수: {len(json_files)}")
    
    for file in json_files:
        try:
            df = load_json_data(file)  
            
            category_info = file.split(os.sep)[-2]  
            df['CategoryPath'] = category_info  
            
            all_data.append(df)  
            
        except Exception as e:
            print(f"Error loading {file}: {e}")
    
    combined_df = pd.concat(all_data, ignore_index=True)
    return combined_df

In [18]:
# ============================================ 
# Cell 3: 데이터 전처리 함수 정의
# ============================================ 

def preprocess_data(df):
    """
    로드된 원시 데이터를 추천 시스템에 적합한 형태로 전처리하는 함수
    
    전처리 과정:
    1. 결측값 처리 (빈 문자열 또는 기본값으로 대체)
    2. 데이터 타입 변환 (문자열을 숫자로 변환 등)
    3. 속성별 감정 점수 추출 및 평균 계산
    4. 텍스트 정제 및 정규화
    
    Parameters:
    -----------
    df : pd.DataFrame
        전처리할 원시 데이터 DataFrame
        
    Returns:
    --------
    pd.DataFrame
        전처리가 완료된 DataFrame
        새로운 'AspectScores' 컬럼이 추가되어 속성별 감정 점수가 딕셔너리 형태로 저장됨
        
    Note:
    -----
    속성 기반 감정분석 데이터의 특수한 구조를 고려하여 설계됨
    """
    print("데이터 전처리 시작...")
    
    # 1단계: 기본 결측값 처리
    df['RawText'] = df['RawText'].fillna('')
    df['ProductName'] = df['ProductName'].fillna('Unknown')
    df['GeneralPolarity'] = pd.to_numeric(df['GeneralPolarity'], errors='coerce').fillna(0)
    
    # 2단계: 속성별 감정 점수를 저장할 새 컬럼 초기화
    df['AspectScores'] = [{}] * len(df)
    
    # 3단계: 속성 기반 감정 점수 추출 (데이터에 Aspects 컬럼이 있는 경우만)
    if 'Aspects' in df.columns:
        print("속성별 감정 점수 추출 중...")
        
        for idx, row in df.iterrows():
            try:
                aspects = row['Aspects']  
                
                if aspects is None or (isinstance(aspects, float) and pd.isna(aspects)):
                    continue
                
                if not isinstance(aspects, list):
                    continue
                
                aspect_scores = {}
                
                for aspect in aspects:
                    if isinstance(aspect, dict):  
                        
                        aspect_name = str(aspect.get('Aspect', '')).strip()
                        
                        try:
                            polarity = float(aspect.get('SentimentPolarity', 0))
                        except (ValueError, TypeError):
                            polarity = 0
                        
                        if aspect_name:
                            if aspect_name in aspect_scores:
                                aspect_scores[aspect_name].append(polarity)
                            else:
                                aspect_scores[aspect_name] = [polarity]
                
                df.at[idx, 'AspectScores'] = {
                    k: np.mean(v) for k, v in aspect_scores.items() if v
                }
                
            except Exception as e:
                continue
    
    print("데이터 전처리 완료!")
    return df

In [ ]:
# ============================================ 
# Cell 4: 실제 데이터 로드 및 전처리 실행
# ============================================ 

print("데이터 로드 시작...")

# 실제 데이터 경로 설정 (사용자 환경에 맞게 수정 필요)
base_path = r'C:\Users\SSAFY\Downloads\147.속성기반 감정분석 데이터\01-1.정식개방데이터'

if os.path.exists(base_path):
    print(f"데이터 경로 확인됨: {base_path}")
    df = load_all_json_files(base_path)  
else:
    print("지정된 경로를 찾을 수 없어 데모용 예시 데이터를 생성합니다...")
    
    # 테스트용 가상 데이터 생성 (실제 데이터가 없을 때 사용)
    sample_data = {
        'ProductName': ['제품A', '제품B', '제품C', '제품D', '제품E'] * 20,  
        'RawText': [
            '정말 좋은 제품입니다. 품질이 뛰어나요.',
            '가격 대비 품질이 우수합니다. 만족스러워요.',
            '디자인이 예쁘고 실용적입니다.',
            '배송도 빠르고 포장도 잘 되어있어요.',
            '다른 사람들에게도 추천하고 싶은 제품입니다.'
        ] * 20,
        'Domain': ['전자제품', '의류', '식품', '화장품', '가전'] * 20,
        'MainCategory': ['스마트폰', '셔츠', '과자', '립스틱', '청소기'] * 20,
        'GeneralPolarity': [0.8, 0.7, 0.9, 0.6, 0.8] * 20,  
        'Source': ['온라인', '오프라인', '온라인', '온라인', '오프라인'] * 20,
        'Aspects': [[{'Aspect': '품질', 'SentimentPolarity': 0.8}]] * 100  
    }
    df = pd.DataFrame(sample_data)
    print(f"예시 데이터 생성 완료: {len(df)}개 행")

# 데이터 전처리 실행
df = preprocess_data(df)

# 추천 시스템을 위한 핵심 데이터 구조 생성
print("추천 시스템용 데이터 구조 생성 중...")

product_texts = df.groupby('ProductName')['RawText'].apply(
    lambda x: ' '.join(x)  
).reset_index()

product_meta = df.groupby('ProductName').agg({
    'Domain': 'first',              
    'MainCategory': 'first',        
    'GeneralPolarity': 'mean',      
    'Source': lambda x: x.mode()[0] if len(x) > 0 else 'Unknown'  
}).reset_index()

print(f"전처리 완료:")
print(f"- RAW 리뷰 데이터: {len(df):,}개")
print(f"- 고유 제품 수: {len(product_texts):,}개")
print(f"- 평균 제품당 리뷰 수: {len(df)/len(product_texts):.1f}개")
print(f"제품 목록 (상위 5개): {product_texts['ProductName'].head().tolist()}")

데이터 로드 시작...
데이터 경로 확인됨: C:\Users\ceo\Downloads\147.속성기반 감정분석 데이터\01-1.정식개방데이터
찾은 JSON 파일 수: 2004
데이터 전처리 시작...
속성별 감정 점수 추출 중...
데이터 전처리 완료!
추천 시스템용 데이터 구조 생성 중...
전처리 완료:
- RAW 리뷰 데이터: 199,733개
- 고유 제품 수: 16,078개
- 평균 제품당 리뷰 수: 12.4개
제품 목록 (상위 5개): [' LG 울트라HD TV 75형(189cm) 75UN7850KNA+LG사운드바', ' [22년 신모델]  LG그램 15Z95P-GA5LK (i5-1155G7/16GB/SSD 256GB/WIN 11)', ' 무선키보드 마우스 세트 저소음 옵티컬 4버튼 해킹방지 ONG ENTUS OGN-WKM20', '(1 + 1) OO 변기세정제 (6개입) ', '(1+1 TJ 태진 블루투스 마이크 / 무선 노래방 마이크']


### 02: TF-IDF 벡터화 기반 콘텐츠 추천
TF-IDF(Term Frequency-Inverse Document Frequency) 알고리즘을 사용하여 제품 간 텍스트 유사도를 계산하는 콘텐츠 기반 추천 시스템을 구현합니다. 

빠른 속도와 해석 용이성이 장점인 전통적인 자연어처리 기법을 활용합니다.

* TFIDFRecommender 클래스 정의 및 TfidfVectorizer 하이퍼파라미터 설정
* 제품별 통합 텍스트에 TF-IDF 벡터화 적용하여 희소 행렬 생성
* 코사인 유사도 계산을 통한 제품 추천 알고리즘 구현
* 추천 다양성, 속도, 커버리지 등 성능 지표 평가 및 결과 출력
* 실제 제품 기준 추천 예시 생성 및 유사도 점수 표시

In [20]:
# ============================================ 
# Cell 1: TF-IDF 추천 시스템 클래스 정의
# ============================================ 

print("\n" + "="*60)
print("2. TF-IDF 적용 및 성능 평가")
print("="*60)

class TFIDFRecommender:
    """
    TF-IDF 벡터화를 기반으로 한 콘텐츠 기반 추천 시스템 클래스
    
    TF-IDF 원리:
    - TF (Term Frequency): 문서 내 단어 출현 빈도
    - IDF (Inverse Document Frequency): 전체 문서에서 단어의 희귀성
    - TF-IDF = TF × IDF (단어가 특정 문서에는 많이, 전체에는 적게 나타날수록 높은 점수)
    
    Attributes:
    -----------
    product_texts : pd.DataFrame
        제품별 통합 텍스트 데이터
    vectorizer : TfidfVectorizer
        scikit-learn의 TF-IDF 벡터라이저
    tfidf_matrix : sparse matrix
        제품-단어 TF-IDF 행렬 (행: 제품, 열: 단어)
    feature_names : array
        벡터라이저가 추출한 단어(특성) 목록
    """
    
    def __init__(self, product_texts):
        """
        TF-IDF 추천 시스템 초기화
        
        Parameters:
        -----------
        product_texts : pd.DataFrame
            제품명과 통합된 리뷰 텍스트가 포함된 DataFrame
            컬럼: ['ProductName', 'RawText']
        """
        self.product_texts = product_texts
        
        self.vectorizer = TfidfVectorizer(
            max_features=1000,      
            ngram_range=(1, 2),     
            min_df=2,               
            max_df=0.8              
        )
        
        self.tfidf_matrix = None    
        self.feature_names = None   
    
    def fit(self):
        """
        TF-IDF 모델 학습 및 벡터 행렬 생성
        
        과정:
        1. 제품별 텍스트에 TF-IDF 벡터화 적용
        2. 희소 행렬 형태로 TF-IDF 값 계산
        3. 벡터라이저에서 특성(단어) 이름 추출
        4. 학습 시간 측정 및 결과 정보 출력
        
        Returns:
        --------
        self : TFIDFRecommender
            메서드 체이닝을 위한 자기 자신 반환
        """
        print("TF-IDF 벡터화 진행 중...")
        start_time = time.time()  
        
        self.tfidf_matrix = self.vectorizer.fit_transform(self.product_texts['RawText'])
        
        self.feature_names = self.vectorizer.get_feature_names_out()
        
        fit_time = time.time() - start_time  
        
        print(f"TF-IDF 학습 완료:")
        print(f"  - 행렬 크기: {self.tfidf_matrix.shape} (제품 수 × 단어 수)")
        print(f"  - 학습 시간: {fit_time:.3f}초")
        print(f"  - 메모리 사용량: {self.tfidf_matrix.data.nbytes / 1024 / 1024:.2f} MB")
        
        return self  
    
    def get_recommendations(self, product_name, n=5):
        """
        특정 제품과 유사한 제품들을 추천하는 함수
        
        추천 알고리즘:
        1. 입력 제품의 TF-IDF 벡터 추출
        2. 모든 제품과의 코사인 유사도 계산
        3. 유사도가 높은 순으로 정렬
        4. 상위 n개 제품 반환 (자기 자신 제외)
        
        Parameters:
        -----------
        product_name : str
            추천을 받을 기준이 되는 제품명
        n : int, default=5
            추천할 제품의 개수
            
        Returns:
        --------
        list of dict
            추천 제품 정보 리스트
            각 딕셔너리는 {'ProductName': 제품명, 'Similarity': 유사도} 형태
        """
        if product_name not in self.product_texts['ProductName'].values:
            print(f"제품 '{product_name}'을 찾을 수 없습니다.")
            return []
        
        idx = self.product_texts[self.product_texts['ProductName'] == product_name].index[0]
        
        product_vector = self.tfidf_matrix[idx:idx+1]  
        
        similarities = cosine_similarity(product_vector, self.tfidf_matrix).flatten()
        
        similar_indices = similarities.argsort()[-n-1:-1][::-1]
        
        recommendations = []
        for i in similar_indices:
            if i != idx:  
                recommendations.append({
                    'ProductName': self.product_texts.iloc[i]['ProductName'],
                    'Similarity': similarities[i]  
                })
        
        return recommendations[:n]  
    
    def evaluate_performance(self, sample_size=20, n_recommendations=5):
        """
        TF-IDF 추천 시스템의 성능을 다각도로 평가하는 함수
        
        평가 지표:
        1. 다양성 (Diversity): 추천된 제품들의 중복도 측정
        2. 속도 (Speed): 추천 생성에 걸리는 평균 시간
        3. 커버리지 (Coverage): 전체 제품 중 추천된 제품의 비율
        4. 메모리 효율성: 희소성(Sparsity) 측정
        
        Parameters:
        -----------
        sample_size : int, default=20
            테스트에 사용할 제품 샘플 수 (전체 평가 시간 단축)
        n_recommendations : int, default=5
            각 제품당 생성할 추천 개수
            
        Returns:
        --------
        dict
            성능 평가 결과가 담긴 딕셔너리
        """
        print("TF-IDF 성능 평가 시작...")
        
        sample_products = self.product_texts['ProductName'].sample(
            min(sample_size, len(self.product_texts)),  
            random_state=42  
        )
        
        all_recommendations = []    
        recommendation_times = []   
        
        for product in sample_products:
            start_time = time.time()  
            recs = self.get_recommendations(product, n_recommendations)
            end_time = time.time()    
            
            recommendation_times.append(end_time - start_time)
            all_recommendations.extend([r['ProductName'] for r in recs])
        
        unique_recommendations = len(set(all_recommendations))  
        total_recommendations = len(all_recommendations)        
        
        diversity = unique_recommendations / total_recommendations if total_recommendations > 0 else 0
        
        avg_time = np.mean(recommendation_times)
        
        sparsity = 1 - (self.tfidf_matrix.nnz / (self.tfidf_matrix.shape[0] * self.tfidf_matrix.shape[1]))
        
        results = {
            'method': 'TF-IDF',
            'diversity': diversity,
            'avg_time': avg_time,
            'unique_recommendations': unique_recommendations,
            'total_recommendations': total_recommendations,
            'matrix_shape': self.tfidf_matrix.shape,
            'sparsity': sparsity,
            'coverage': unique_recommendations / len(self.product_texts) if len(self.product_texts) > 0 else 0
        }
        
        print(f"TF-IDF 성능 평가 결과:")
        print(f"  - 추천 다양성: {diversity:.3f} (1에 가까울수록 다양)")
        print(f"  - 평균 추천 시간: {avg_time:.4f}초")
        print(f"  - 데이터 희소성: {sparsity:.3f} (높을수록 메모리 효율적)")
        print(f"  - 추천 커버리지: {results['coverage']:.3f}")
        print(f"  - 고유 추천 제품: {unique_recommendations}/{total_recommendations}")
        
        return results


2. TF-IDF 적용 및 성능 평가


In [21]:
# ============================================ 
# Cell 2: TF-IDF 추천 시스템 실행 및 평가
# ============================================ 

print("TF-IDF 추천 시스템 초기화...")
tfidf_recommender = TFIDFRecommender(product_texts)

tfidf_recommender.fit()

tfidf_results = tfidf_recommender.evaluate_performance()

if len(product_texts) > 0:
    sample_product = product_texts['ProductName'].iloc[0]  
    sample_recs = tfidf_recommender.get_recommendations(sample_product, 3)
    
    print(f"\n추천 예시 - '{sample_product}' 기준:")
    print("-" * 50)
    for i, rec in enumerate(sample_recs, 1):
        print(f"  {i}. {rec['ProductName']}")
        print(f"     유사도: {rec['Similarity']:.3f}")
    
    if not sample_recs:
        print("  추천 결과가 없습니다.")

TF-IDF 추천 시스템 초기화...
TF-IDF 벡터화 진행 중...
TF-IDF 학습 완료:
  - 행렬 크기: (16078, 1000) (제품 수 × 단어 수)
  - 학습 시간: 13.865초
  - 메모리 사용량: 7.87 MB
TF-IDF 성능 평가 시작...
TF-IDF 성능 평가 결과:
  - 추천 다양성: 0.990 (1에 가까울수록 다양)
  - 평균 추천 시간: 0.0187초
  - 데이터 희소성: 0.936 (높을수록 메모리 효율적)
  - 추천 커버리지: 0.006
  - 고유 추천 제품: 99/100

추천 예시 - ' LG 울트라HD TV 75형(189cm) 75UN7850KNA+LG사운드바' 기준:
--------------------------------------------------
  1. 삼성전자 갤럭시탭S8 울트라 Wi-Fi 128GB 정품
     유사도: 0.507
  2. 삼성전자 갤럭시탭S7 FE Wi-Fi 64GB 정품
     유사도: 0.469
  3. Crystal UHD 삼성전자 KU85UA7000FXKR 스탠드
     유사도: 0.454


### 03: LSA 차원 축소를 통한 추천
LSA(Latent Semantic Analysis)를 통해 고차원 TF-IDF 벡터를 저차원으로 축소하여 잠재적 의미 관계를 포착합니다. 

SVD 분해를 사용하여 노이즈를 감소시키고 동의어나 유사 개념을 효과적으로 처리하는 추천 시스템을 구현합니다.

* LSARecommender 클래스 정의 및 TruncatedSVD를 사용한 차원 축소 설정
* TF-IDF 행렬에 SVD 적용하여 지정된 차원 수로 잠재 의미 공간 생성
* LSA 공간에서 코사인 유사도 기반 제품 추천 알고리즘 구현
* 설명 분산 비율 분석 및 차원 축소 효과 정량적 평가
* Scree Plot과 누적 설명 분산 시각화로 최적 차원 수 분석

In [22]:
# ============================================ 
# Cell 1: LSA 추천 시스템 클래스 정의
# ============================================ 

print("\n" + "="*60)
print("3. LSA (Latent Semantic Analysis) 적용 및 성능 평가")
print("="*60)

class LSARecommender:
    """
    LSA(잠재 의미 분석) 기반 콘텐츠 추천 시스템 클래스
    
    LSA 동작 원리:
    1. TF-IDF 행렬을 SVD로 분해: A = U × Σ × V^T
    2. 상위 k개의 특이값만 유지하여 차원 축소
    3. 축소된 공간에서 문서 간 유사도 계산
    4. 의미적으로 유사한 단어들이 가까운 위치에 배치됨
    
    장점:
    - 동의어, 유사어 관계 포착 가능
    - 차원 축소를 통한 노이즈 제거
    - 계산 복잡도 감소
    
    단점:
    - 해석이 어려움 (잠재 차원의 의미가 불분명)
    - 음수 값 가능
    """
    
    def __init__(self, product_texts, n_components=50):
        """
        LSA 추천 시스템 초기화
        
        Parameters:
        -----------
        product_texts : pd.DataFrame
            제품별 통합 텍스트 데이터
        n_components : int, default=50
            축소할 차원 수 (잠재 토픽 수)
            - 너무 낮으면: 정보 손실 심각
            - 너무 높으면: 노이즈 포함, 계산 비용 증가
            - 일반적으로 50-200 사이 권장
        """
        self.product_texts = product_texts
        self.n_components = n_components
        
        self.vectorizer = TfidfVectorizer(
            max_features=1000,      
            ngram_range=(1, 2),     
            min_df=2,               
            max_df=0.8              
        )
        
        self.svd = TruncatedSVD(
            n_components=n_components, 
            random_state=42  
        )
        
        self.tfidf_matrix = None    
        self.lsa_matrix = None      
    
    def fit(self):
        """
        LSA 모델 학습 및 차원 축소 수행
        
        학습 과정:
        1. 텍스트를 TF-IDF 벡터로 변환
        2. TF-IDF 행렬에 SVD 적용
        3. 상위 n_components개 차원으로 축소
        4. 설명 분산 비율 계산 및 출력
        
        Returns:
        --------
        self : LSARecommender
            메서드 체이닝을 위한 자기 자신 반환
        """
        print(f"LSA 학습 시작 (목표 차원: {self.n_components})...")
        start_time = time.time()
        
        print("  Step 1: TF-IDF 벡터화...")
        self.tfidf_matrix = self.vectorizer.fit_transform(self.product_texts['RawText'])
        
        print("  Step 2: SVD 차원 축소...")
        self.lsa_matrix = self.svd.fit_transform(self.tfidf_matrix)
        
        fit_time = time.time() - start_time
        
        explained_variance_ratio = self.svd.explained_variance_ratio_  
        cumulative_variance = np.cumsum(explained_variance_ratio)      
        
        print(f"LSA 학습 완료:")
        print(f"  - 원본 차원: {self.tfidf_matrix.shape[1]:,}차원")
        print(f"  - 축소 차원: {self.lsa_matrix.shape[1]:,}차원")
        print(f"  - 학습 시간: {fit_time:.3f}초")
        print(f"  - 총 설명 분산: {cumulative_variance[-1]:.3f}")
        print(f"  - 상위 5개 차원의 설명 분산: {explained_variance_ratio[:5]}")
        
        dim_for_80_percent = np.argmax(cumulative_variance >= 0.8) + 1
        if cumulative_variance[-1] >= 0.8:
            print(f"  - 80% 분산 설명 차원: {dim_for_80_percent}개")
        else:
            print(f"  - 현재 차원으로는 {cumulative_variance[-1]*100:.1f}% 분산만 설명 가능")
        
        return self
    
    def get_recommendations(self, product_name, n=5):
        """
        LSA 공간에서 유사한 제품 추천
        
        추천 과정:
        1. 입력 제품의 LSA 벡터 추출
        2. 모든 제품과의 코사인 유사도 계산 (LSA 공간에서)
        3. 유사도 기준 상위 n개 제품 반환
        
        Parameters:
        -----------
        product_name : str
            기준 제품명
        n : int, default=5
            추천할 제품 수
            
        Returns:
        --------
        list of dict
            추천 제품 정보 리스트
        """
        if product_name not in self.product_texts['ProductName'].values:
            print(f"제품 '{product_name}'을 찾을 수 없습니다.")
            return []
        
        idx = self.product_texts[self.product_texts['ProductName'] == product_name].index[0]
        
        product_vector = self.lsa_matrix[idx:idx+1]
        
        similarities = cosine_similarity(product_vector, self.lsa_matrix).flatten()
        
        similar_indices = similarities.argsort()[-n-1:-1][::-1]
        
        recommendations = []
        for i in similar_indices:
            if i != idx:  
                recommendations.append({
                    'ProductName': self.product_texts.iloc[i]['ProductName'],
                    'Similarity': similarities[i]
                })
        
        return recommendations[:n]
    
    def evaluate_performance(self, sample_size=20, n_recommendations=5):
        """
        LSA 추천 시스템의 성능 평가
        
        LSA 특화 평가 항목:
        1. 기본 성능 지표 (다양성, 속도)
        2. 차원 축소 효과 (설명 분산)
        3. 의미적 유사성 품질
        
        Parameters:
        -----------
        sample_size : int, default=20
            평가용 샘플 크기
        n_recommendations : int, default=5
            제품당 추천 수
            
        Returns:
        --------
        dict
            LSA 성능 평가 결과
        """
        print("LSA 성능 평가 시작...")
        
        sample_products = self.product_texts['ProductName'].sample(
            min(sample_size, len(self.product_texts)), 
            random_state=42
        )
        
        all_recommendations = []
        recommendation_times = []
        
        for product in sample_products:
            start_time = time.time()
            recs = self.get_recommendations(product, n_recommendations)
            end_time = time.time()
            
            recommendation_times.append(end_time - start_time)
            all_recommendations.extend([r['ProductName'] for r in recs])
        
        unique_recommendations = len(set(all_recommendations))
        total_recommendations = len(all_recommendations)
        diversity = unique_recommendations / total_recommendations if total_recommendations > 0 else 0
        avg_time = np.mean(recommendation_times)
        
        explained_variance = np.sum(self.svd.explained_variance_ratio_)  
        
        results = {
            'method': 'LSA',
            'diversity': diversity,
            'avg_time': avg_time,
            'unique_recommendations': unique_recommendations,
            'total_recommendations': total_recommendations,
            'matrix_shape': self.lsa_matrix.shape,
            'explained_variance': explained_variance,
            'n_components': self.n_components,
            'dimension_reduction_ratio': self.lsa_matrix.shape[1] / self.tfidf_matrix.shape[1]
        }
        
        print(f"LSA 성능 평가 결과:")
        print(f"  - 추천 다양성: {diversity:.3f}")
        print(f"  - 평균 추천 시간: {avg_time:.4f}초")
        print(f"  - 설명 분산: {explained_variance:.3f}")
        print(f"  - 차원 축소 비율: {results['dimension_reduction_ratio']:.3f}")
        print(f"  - 축소된 차원 수: {self.n_components}")
        
        return results
    
    def visualize_explained_variance(self):
        """
        LSA의 설명 분산을 시각화하여 최적 차원 수 분석
        
        시각화 내용:
        1. 개별 차원별 설명 분산 (Scree Plot)
        2. 누적 설명 분산 (80% 기준선 포함)
        
        이를 통해 정보 손실 없이 차원을 얼마나 축소할 수 있는지 판단 가능
        """
        if self.svd is None:
            print("먼저 fit() 메서드를 실행해주세요.")
            return
        
        explained_var = self.svd.explained_variance_ratio_
        cumulative_var = np.cumsum(explained_var)
        
        # 테이블 형식으로 출력
        print("\n" + "="*60)
        print("LSA 설명 분산 분석")
        print("="*60)
        
        print("\n개별 차원별 설명 분산 (상위 10개):")
        print(f"{'차원':^10} | {'설명 분산':^15} | {'누적 분산':^15}")
        print("-" * 45)
        for i in range(min(10, len(explained_var))):
            print(f"{i+1:^10} | {explained_var[i]:^15.4f} | {cumulative_var[i]:^15.4f}")
        
        # 주요 통계
        idx_80 = np.argmax(cumulative_var >= 0.8)
        idx_90 = np.argmax(cumulative_var >= 0.9)
        
        print(f"\nLSA 차원 선택 가이드:")
        print(f"  - 현재 사용 차원: {self.n_components}")
        print(f"  - 현재 설명 분산: {cumulative_var[-1]:.3f}")
        if cumulative_var[-1] >= 0.8:
            print(f"  - 80% 분산 달성 차원: {idx_80+1}")
        if len(cumulative_var) > idx_90 and cumulative_var[idx_90] >= 0.9:
            print(f"  - 90% 분산 달성 차원: {idx_90+1}")


3. LSA (Latent Semantic Analysis) 적용 및 성능 평가


In [23]:
# ============================================ 
# Cell 2: LSA 추천 시스템 실행 및 평가
# ============================================ 

print("LSA 추천 시스템 초기화 및 학습...")
lsa_recommender = LSARecommender(product_texts, n_components=50)
lsa_recommender.fit()

lsa_results = lsa_recommender.evaluate_performance()

lsa_recommender.visualize_explained_variance()

if len(product_texts) > 0:
    sample_product = product_texts['ProductName'].iloc[0]
    sample_recs = lsa_recommender.get_recommendations(sample_product, 3)
    
    print(f"\nLSA 추천 예시 - '{sample_product}' 기준:")
    print("-" * 50)
    for i, rec in enumerate(sample_recs, 1):
        print(f"  {i}. {rec['ProductName']}")
        print(f"     유사도: {rec['Similarity']:.3f} (LSA 공간)")
    
    if not sample_recs:
        print("  추천 결과가 없습니다.")


LSA 추천 시스템 초기화 및 학습...
LSA 학습 시작 (목표 차원: 50)...
  Step 1: TF-IDF 벡터화...
  Step 2: SVD 차원 축소...
LSA 학습 완료:
  - 원본 차원: 1,000차원
  - 축소 차원: 50차원
  - 학습 시간: 14.312초
  - 총 설명 분산: 0.235
  - 상위 5개 차원의 설명 분산: [0.01907083 0.01751683 0.01201141 0.01055641 0.00880472]
  - 현재 차원으로는 23.5% 분산만 설명 가능
LSA 성능 평가 시작...
LSA 성능 평가 결과:
  - 추천 다양성: 0.990
  - 평균 추천 시간: 0.0070초
  - 설명 분산: 0.235
  - 차원 축소 비율: 0.050
  - 축소된 차원 수: 50

LSA 설명 분산 분석

개별 차원별 설명 분산 (상위 10개):
    차원     |      설명 분산      |      누적 분산     
---------------------------------------------
    1      |     0.0191      |     0.0191     
    2      |     0.0175      |     0.0366     
    3      |     0.0120      |     0.0486     
    4      |     0.0106      |     0.0592     
    5      |     0.0088      |     0.0680     
    6      |     0.0075      |     0.0754     
    7      |     0.0069      |     0.0823     
    8      |     0.0067      |     0.0890     
    9      |     0.0064      |     0.0954     
    10     |     0.0057      |     0

### 04: PCA 기반 추천 시스템
PCA(Principal Component Analysis)를 활용하여 데이터의 분산을 최대한 보존하면서 차원을 축소하는 통계적 방법으로 추천 시스템을 구현합니다. 

주성분들의 선형 결합을 통해 해석 가능한 특징 공간에서 제품 유사도를 계산합니다.

* PCARecommender 클래스 정의 및 StandardScaler를 사용한 데이터 정규화 설정
* 희소 행렬을 밀집 행렬로 변환 후 PCA 적용하여 주성분 공간 생성
* PCA 공간에서 코사인 유사도 기반 제품 추천 및 성능 평가
* 주성분별 설명 분산 분석 및 차원 압축 효과 계산
* 첫 두 주성분 공간에서의 데이터 분포 및 주성분 기여도 시각화

In [24]:
# ============================================ 
# Cell 1: PCA 추천 시스템 클래스 정의
# ============================================ 

print("\n" + "="*60)
print("4. PCA (Principal Component Analysis) 적용 및 성능 평가")
print("="*60)

class PCARecommender:
    """
    PCA(주성분 분석) 기반 콘텐츠 추천 시스템 클래스
    
    PCA 동작 원리:
    1. 데이터의 공분산 행렬 계산
    2. 고유값과 고유벡터 추출
    3. 고유값이 큰 순서대로 주성분 선택
    4. 원본 데이터를 주성분 공간으로 투영
    
    장점:
    - 분산 보존 최적화
    - 주성분의 기여도 명확
    - 선형 변환으로 해석 용이
    
    단점:
    - 텍스트 데이터의 의미적 관계 포착 제한
    - 모든 특성이 주성분에 기여 (희소성 손실)
    """
    
    def __init__(self, product_texts, n_components=50):
        """
        PCA 추천 시스템 초기화
        
        Parameters:
        -----------
        product_texts : pd.DataFrame
            제품별 통합 텍스트 데이터
        n_components : int, default=50
            주성분 개수 (축소할 차원 수)
            - 원본 특성 수보다 작아야 함
            - 일반적으로 분산의 80-90%를 설명하는 수준 선택
        """
        self.product_texts = product_texts
        self.n_components = n_components
        
        self.vectorizer = TfidfVectorizer(
            max_features=1000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.8
        )
        
        self.scaler = StandardScaler(with_mean=False)
        
        self.pca = PCA(
            n_components=n_components, 
            random_state=42  
        )
        
        self.tfidf_matrix = None    
        self.pca_matrix = None      
    
    def fit(self):
        """
        PCA 모델 학습 및 차원 축소 수행
        
        학습 과정:
        1. TF-IDF 벡터화
        2. 희소 행렬을 밀집 행렬로 변환
        3. 표준화 (스케일링)
        4. PCA 적용 및 차원 축소
        5. 주성분 분석 결과 출력
        
        Returns:
        --------
        self : PCARecommender
            메서드 체이닝용 자기 자신 반환
        """
        print(f"PCA 학습 시작 (목표 주성분: {self.n_components}개)...")
        start_time = time.time()
        
        print("  Step 1: TF-IDF 벡터화...")
        self.tfidf_matrix = self.vectorizer.fit_transform(self.product_texts['RawText'])
        
        print("  Step 2: 희소 행렬 -> 밀집 행렬 변환...")
        dense_matrix = self.tfidf_matrix.toarray()
        
        print("  Step 3: 데이터 표준화...")
        scaled_matrix = self.scaler.fit_transform(dense_matrix)
        
        print("  Step 4: PCA 차원 축소...")
        self.pca_matrix = self.pca.fit_transform(scaled_matrix)
        
        fit_time = time.time() - start_time
        
        explained_variance_ratio = self.pca.explained_variance_ratio_
        cumulative_variance = np.cumsum(explained_variance_ratio)
        
        print(f"PCA 학습 완료:")
        print(f"  - 원본 특성 수: {dense_matrix.shape[1]:,}개")
        print(f"  - 주성분 수: {self.pca_matrix.shape[1]:,}개")
        print(f"  - 학습 시간: {fit_time:.3f}초")
        print(f"  - 총 설명 분산: {cumulative_variance[-1]:.3f}")
        print(f"  - 1st 주성분 설명 분산: {explained_variance_ratio[0]:.3f}")
        print(f"  - 상위 5개 주성분 분산: {explained_variance_ratio[:5]}")
        
        compression_ratio = self.pca_matrix.shape[1] / dense_matrix.shape[1]
        print(f"  - 차원 압축 비율: {compression_ratio:.3f}")
        print(f"  - 메모리 절약: {(1-compression_ratio)*100:.1f}%")
        
        return self
    
    def get_recommendations(self, product_name, n=5):
        """
        PCA 공간에서 유사한 제품 추천
        
        PCA 공간에서의 유사도는 주성분들로 구성된 새로운 좌표계에서 계산됨
        각 주성분은 원본 특성들의 선형 결합으로 구성됨
        
        Parameters:
        -----------
        product_name : str
            기준 제품명
        n : int, default=5
            추천할 제품 수
            
        Returns:
        --------
        list of dict
            추천 제품 정보 리스트
        """
        if product_name not in self.product_texts['ProductName'].values:
            print(f"제품 '{product_name}'을 찾을 수 없습니다.")
            return []
        
        idx = self.product_texts[self.product_texts['ProductName'] == product_name].index[0]
        
        product_vector = self.pca_matrix[idx:idx+1]
        
        similarities = cosine_similarity(product_vector, self.pca_matrix).flatten()
        
        similar_indices = similarities.argsort()[-n-1:-1][::-1]
        
        recommendations = []
        for i in similar_indices:
            if i != idx:  
                recommendations.append({
                    'ProductName': self.product_texts.iloc[i]['ProductName'],
                    'Similarity': similarities[i]
                })
        
        return recommendations[:n]
    
    def evaluate_performance(self, sample_size=20, n_recommendations=5):
        """
        PCA 추천 시스템의 성능 평가
        
        PCA 특화 평가 항목:
        1. 기본 성능 지표 (다양성, 속도)
        2. 주성분 분석 품질 (설명 분산)
        3. 차원 축소 효과
        
        Parameters:
        -----------
        sample_size : int, default=20
            평가용 샘플 크기
        n_recommendations : int, default=5
            제품당 추천 수
            
        Returns:
        --------
        dict
            PCA 성능 평가 결과
        """
        print("PCA 성능 평가 시작...")
        
        sample_products = self.product_texts['ProductName'].sample(
            min(sample_size, len(self.product_texts)), 
            random_state=42
        )
        
        all_recommendations = []
        recommendation_times = []
        
        for product in sample_products:
            start_time = time.time()
            recs = self.get_recommendations(product, n_recommendations)
            end_time = time.time()
            
            recommendation_times.append(end_time - start_time)
            all_recommendations.extend([r['ProductName'] for r in recs])
        
        unique_recommendations = len(set(all_recommendations))
        total_recommendations = len(all_recommendations)
        diversity = unique_recommendations / total_recommendations if total_recommendations > 0 else 0
        avg_time = np.mean(recommendation_times)
        
        explained_variance = np.sum(self.pca.explained_variance_ratio_)
        
        results = {
            'method': 'PCA',
            'diversity': diversity,
            'avg_time': avg_time,
            'unique_recommendations': unique_recommendations,
            'total_recommendations': total_recommendations,
            'matrix_shape': self.pca_matrix.shape,
            'explained_variance': explained_variance,
            'n_components': self.n_components,
            'first_pc_variance': self.pca.explained_variance_ratio_[0]  
        }
        
        print(f"PCA 성능 평가 결과:")
        print(f"  - 추천 다양성: {diversity:.3f}")
        print(f"  - 평균 추천 시간: {avg_time:.4f}초")
        print(f"  - 설명 분산: {explained_variance:.3f}")
        print(f"  - 1st 주성분 기여도: {results['first_pc_variance']:.3f}")
        print(f"  - 주성분 수: {self.n_components}")
        
        return results
    
    def visualize_components(self):
        """
        PCA 주성분 분석 결과를 다각도로 시각화
        
        시각화 내용:
        1. 개별 주성분별 설명 분산
        2. 누적 설명 분산 (80% 기준선)
        3. 첫 두 주성분 공간에서의 데이터 분포
        
        이를 통해 주성분의 중요도와 데이터의 분포 패턴 파악 가능
        """
        if self.pca is None:
            print("먼저 fit() 메서드를 실행해주세요.")
            return
        
        explained_var = self.pca.explained_variance_ratio_
        cumulative_var = np.cumsum(explained_var)
        
        # 테이블 형식으로 출력
        print("\n" + "="*60)
        print("PCA 주성분 분석 결과")
        print("="*60)
        
        print("\n개별 주성분별 설명 분산 (상위 10개):")
        print(f"{'주성분':^10} | {'설명 분산':^15} | {'누적 분산':^15}")
        print("-" * 45)
        for i in range(min(10, len(explained_var))):
            print(f"{i+1:^10} | {explained_var[i]:^15.4f} | {cumulative_var[i]:^15.4f}")
        
        # 주요 통계
        idx_80 = np.argmax(cumulative_var >= 0.8)
        idx_90 = np.argmax(cumulative_var >= 0.9)
        
        print(f"\nPCA 주성분 분석 결과:")
        print(f"  - 현재 주성분 수: {self.n_components}")
        print(f"  - 현재 설명 분산: {cumulative_var[-1]:.3f}")
        print(f"  - 1st 주성분: {explained_var[0]:.3f} (가장 큰 분산 방향)")
        print(f"  - 2nd 주성분: {explained_var[1]:.3f} (1st와 직교하는 최대 분산 방향)")
        
        if cumulative_var[-1] >= 0.8:
            print(f"  - 80% 분산 달성: {idx_80+1}개 주성분 필요")
        else:
            print(f"  - 권장: 더 많은 주성분으로 80% 이상 분산 확보")
        
        # 첫 두 주성분 공간의 데이터 분포 통계
        print(f"\n첫 두 주성분 공간 데이터 분포:")
        print(f"  - PC1 범위: [{self.pca_matrix[:, 0].min():.3f}, {self.pca_matrix[:, 0].max():.3f}]")
        print(f"  - PC2 범위: [{self.pca_matrix[:, 1].min():.3f}, {self.pca_matrix[:, 1].max():.3f}]")
        print(f"  - PC1 표준편차: {self.pca_matrix[:, 0].std():.3f}")
        print(f"  - PC2 표준편차: {self.pca_matrix[:, 1].std():.3f}")


4. PCA (Principal Component Analysis) 적용 및 성능 평가


In [25]:
# ============================================ 
# Cell 2: PCA 추천 시스템 실행 및 평가
# ============================================ 

print("PCA 추천 시스템 초기화 및 학습...")
pca_recommender = PCARecommender(product_texts, n_components=50)
pca_recommender.fit()

pca_results = pca_recommender.evaluate_performance()

pca_recommender.visualize_components()

if len(product_texts) > 0:
    sample_product = product_texts['ProductName'].iloc[0]
    sample_recs = pca_recommender.get_recommendations(sample_product, 3)
    
    print(f"\nPCA 추천 예시 - '{sample_product}' 기준:")
    print("-" * 50)
    for i, rec in enumerate(sample_recs, 1):
        print(f"  {i}. {rec['ProductName']}")
        print(f"     유사도: {rec['Similarity']:.3f} (PCA 공간)")
    
    if not sample_recs:
        print("  추천 결과가 없습니다.")

PCA 추천 시스템 초기화 및 학습...
PCA 학습 시작 (목표 주성분: 50개)...
  Step 1: TF-IDF 벡터화...
  Step 2: 희소 행렬 -> 밀집 행렬 변환...
  Step 3: 데이터 표준화...
  Step 4: PCA 차원 축소...
PCA 학습 완료:
  - 원본 특성 수: 1,000개
  - 주성분 수: 50개
  - 학습 시간: 14.573초
  - 총 설명 분산: 0.145
  - 1st 주성분 설명 분산: 0.013
  - 상위 5개 주성분 분산: [0.01251139 0.00995369 0.00739714 0.00680089 0.00498571]
  - 차원 압축 비율: 0.050
  - 메모리 절약: 95.0%
PCA 성능 평가 시작...
PCA 성능 평가 결과:
  - 추천 다양성: 0.990
  - 평균 추천 시간: 0.0075초
  - 설명 분산: 0.145
  - 1st 주성분 기여도: 0.013
  - 주성분 수: 50

PCA 주성분 분석 결과

개별 주성분별 설명 분산 (상위 10개):
   주성분     |      설명 분산      |      누적 분산     
---------------------------------------------
    1      |     0.0125      |     0.0125     
    2      |     0.0100      |     0.0225     
    3      |     0.0074      |     0.0299     
    4      |     0.0068      |     0.0367     
    5      |     0.0050      |     0.0416     
    6      |     0.0044      |     0.0461     
    7      |     0.0038      |     0.0499     
    8      |     0.0036      |     0.0535  

### 05: K-means 클러스터링을 통한 특징 공간 품질 평가
각 차원 축소 방법으로 생성된 특징 공간에서 K-means 클러스터링을 수행하여 클러스터 품질을 평가합니다. 

Silhouette Score, Calinski-Harabasz Score 등의 지표를 통해 특징 공간의 우수성을 간접적으로 비교 분석합니다.

* 모든 추천 방법의 특징 행렬 수집 및 K-means 클러스터링 수행
* Silhouette, Calinski-Harabasz, Davies-Bouldin Score로 클러스터 품질 정량 평가
* 클러스터 크기 분포 및 균형도 분석을 통한 데이터 구조 특성 파악
* 방법별 클러스터링 성능 비교 시각화 및 종합 성능 점수 계산
* 클러스터링 시간과 품질을 종합한 특징 공간 우수성 순위 도출

In [26]:
# ============================================ 
# Cell 1: 클러스터링 평가 함수 정의
# ============================================ 

print("\n" + "="*60)
print("5. K-means 클러스터링 적용 및 성능 평가")
print("="*60)

def perform_clustering_evaluation(recommenders, n_clusters=5):
    """
    모든 추천 방법의 특징 공간에서 K-means 클러스터링을 수행하고 품질을 비교 평가
    
    클러스터링 평가 지표 설명:
    1. Silhouette Score: 클러스터 내 응집도와 클러스터 간 분리도의 조화 평균
       - 범위: -1 ~ 1, 높을수록 좋음
       - 0.7+: 강한 구조, 0.5+: 합리적 구조, 0.25+: 약한 구조
    
    2. Calinski-Harabasz Score: 클러스터 간 분산 / 클러스터 내 분산
       - 범위: 0+, 높을수록 좋음
       - 클러스터 간 분리가 명확할수록 높은 값
    
    3. Davies-Bouldin Score: 클러스터 간 평균 유사도
       - 범위: 0+, 낮을수록 좋음
       - 잘 분리된 클러스터일수록 낮은 값
    
    Parameters:
    -----------
    recommenders : dict
        각 추천 방법의 인스턴스들을 담은 딕셔너리
    n_clusters : int, default=5
        생성할 클러스터의 수
        - 데이터 크기와 도메인 특성에 따라 조정
        - 너무 적으면: 클러스터가 너무 일반적
        - 너무 많으면: 의미 없는 세분화
    
    Returns:
    --------
    dict
        각 방법별 클러스터링 결과와 평가 지표
    """
    print(f"K-means 클러스터링 평가 시작 (클러스터 수: {n_clusters})...")
    
    clustering_results = {}
    
    print("특징 행렬 수집 중...")
    feature_matrices = {
        'TF-IDF': tfidf_recommender.tfidf_matrix.toarray(),  
        'LSA': lsa_recommender.lsa_matrix,                   
        'PCA': pca_recommender.pca_matrix                   
    }
    
    for method_name, features in feature_matrices.items():
        print(f"\n{method_name} 특징 공간 클러스터링...")
        print(f"  - 특징 행렬 크기: {features.shape}")
        
        start_time = time.time()
        kmeans = KMeans(
            n_clusters=n_clusters,    
            random_state=42,         
            n_init=10,              
            max_iter=300            
        )
        labels = kmeans.fit_predict(features)
        clustering_time = time.time() - start_time
        
        print("  - 평가 지표 계산 중...")
        
        try:
            silhouette = silhouette_score(features, labels)
        except Exception as e:
            print(f"    Silhouette 계산 오류: {e}")
            silhouette = -1
        
        try:
            calinski = calinski_harabasz_score(features, labels)
        except Exception as e:
            print(f"    Calinski-Harabasz 계산 오류: {e}")
            calinski = 0
        
        try:
            davies_bouldin = davies_bouldin_score(features, labels)
        except Exception as e:
            print(f"    Davies-Bouldin 계산 오류: {e}")
            davies_bouldin = float('inf')
        
        unique_labels, counts = np.unique(labels, return_counts=True)
        cluster_sizes = dict(zip(unique_labels, counts))
        
        cluster_probs = counts / counts.sum()
        cluster_entropy = -np.sum(cluster_probs * np.log2(cluster_probs + 1e-10))
        max_entropy = np.log2(n_clusters)  
        balance_ratio = cluster_entropy / max_entropy if max_entropy > 0 else 0
        
        clustering_results[method_name] = {
            'labels': labels,                    
            'silhouette': silhouette,           
            'calinski': calinski,               
            'davies_bouldin': davies_bouldin,   
            'cluster_sizes': cluster_sizes,      
            'balance_ratio': balance_ratio,      
            'clustering_time': clustering_time,  
            'inertia': kmeans.inertia_,         
            'n_clusters': n_clusters
        }
        
        print(f"  클러스터링 결과:")
        print(f"    Silhouette Score: {silhouette:.3f} (높을수록 좋음)")
        print(f"    Calinski-Harabasz: {calinski:.3f} (높을수록 좋음)")
        print(f"    Davies-Bouldin: {davies_bouldin:.3f} (낮을수록 좋음)")
        print(f"    클러스터 균형도: {balance_ratio:.3f} (1에 가까울수록 균등)")
        print(f"    클러스터링 시간: {clustering_time:.3f}초")
        print(f"    클러스터 크기: {cluster_sizes}")
        
        if silhouette > 0.7:
            quality = "매우 우수"
        elif silhouette > 0.5:
            quality = "우수"
        elif silhouette > 0.25:
            quality = "보통"
        else:
            quality = "부족"
        print(f"    종합 평가: {quality}")
    
    return clustering_results

def visualize_clustering_results(clustering_results):
    """
    클러스터링 결과를 종합적으로 시각화하여 방법 간 비교 분석
    
    시각화 구성:
    1. Silhouette Score 비교 (클러스터 품질)
    2. Calinski-Harabasz Score 비교 (클러스터 분리도)
    3. Davies-Bouldin Score 비교 (클러스터 응집도)
    4. 클러스터 크기 분포 (첫 번째 방법)
    5. 종합 성능 점수
    6. 클러스터링 시간 비교
    
    Parameters:
    -----------
    clustering_results : dict
        각 방법의 클러스터링 평가 결과
    
    Returns:
    --------
    tuple
        (최고 성능 방법명, 각 방법의 종합 점수 배열)
    """
    print("\n클러스터링 결과 종합 분석...")
    
    methods = list(clustering_results.keys())
    
    # 테이블 형식으로 출력
    print("\n" + "="*100)
    print("클러스터링 품질 비교 분석")
    print("="*100)
    
    # 성능 지표 테이블
    print(f"\n{'방법':^10} | {'Silhouette':^12} | {'Calinski-H':^12} | {'Davies-B':^12} | {'균형도':^10} | {'시간(초)':^10}")
    print("-" * 80)
    
    silhouette_scores = []
    calinski_scores = []
    davies_scores = []
    
    for method in methods:
        result = clustering_results[method]
        silhouette_scores.append(result['silhouette'])
        calinski_scores.append(result['calinski'])
        davies_scores.append(result['davies_bouldin'])
        
        print(f"{method:^10} | {result['silhouette']:^12.3f} | {result['calinski']:^12.1f} | "
              f"{result['davies_bouldin']:^12.3f} | {result['balance_ratio']:^10.3f} | "
              f"{result['clustering_time']:^10.3f}")
    
    # 종합 점수 계산
    print("\n종합 성능 점수 계산...")
    
    # 정규화
    if max(silhouette_scores) > 0:
        norm_silhouette = np.array(silhouette_scores) / max(silhouette_scores)
    else:
        norm_silhouette = np.zeros(len(silhouette_scores))
    
    if max(calinski_scores) > 0:
        norm_calinski = np.array(calinski_scores) / max(calinski_scores)
    else:
        norm_calinski = np.zeros(len(calinski_scores))
    
    if max(davies_scores) > 0:
        norm_davies = 1 - (np.array(davies_scores) / max(davies_scores))
    else:
        norm_davies = np.ones(len(davies_scores))
    
    overall_scores = (norm_silhouette + norm_calinski + norm_davies) / 3
    
    # 종합 점수 테이블
    print(f"\n{'방법':^10} | {'종합 점수':^12} | {'순위':^8}")
    print("-" * 35)
    
    sorted_indices = np.argsort(overall_scores)[::-1]
    for rank, idx in enumerate(sorted_indices, 1):
        method = methods[idx]
        score = overall_scores[idx]
        star = "★" if rank == 1 else ""
        print(f"{method:^10} | {score:^12.3f} | {rank:^8} {star}")
    
    best_idx = np.argmax(overall_scores)
    best_method = methods[best_idx]
    
    # 클러스터 크기 분포 (첫 번째 방법)
    print(f"\n{methods[0]} 클러스터 크기 분포:")
    cluster_sizes = clustering_results[methods[0]]['cluster_sizes']
    total_size = sum(cluster_sizes.values())
    for cluster_id, size in cluster_sizes.items():
        percentage = (size / total_size) * 100
        print(f"  클러스터 {cluster_id + 1}: {size}개 ({percentage:.1f}%)")
    
    print(f"\n클러스터링 성능 요약:")
    print(f"   최고 종합 성능: {best_method} (점수: {overall_scores[best_idx]:.3f})")
    print(f"   최고 품질: {methods[np.argmax(silhouette_scores)]} (Silhouette: {max(silhouette_scores):.3f})")
    print(f"   최고 분리도: {methods[np.argmax(calinski_scores)]} (Calinski: {max(calinski_scores):.1f})")
    print(f"   최고 응집도: {methods[np.argmin(davies_scores)]} (Davies-Bouldin: {min(davies_scores):.3f})")
    
    clustering_times = [clustering_results[m]['clustering_time'] for m in methods]
    fastest_idx = np.argmin(clustering_times)
    print(f"   최고 속도: {methods[fastest_idx]} ({min(clustering_times):.3f}초)")
    
    return best_method, overall_scores


5. K-means 클러스터링 적용 및 성능 평가


In [27]:
# ============================================ 
# Cell 2: 클러스터링 평가 실행
# ============================================ 

print("="*60)
print("모든 방법에 대한 클러스터링 품질 평가")
print("="*60)

recommender_instances = {
    'tfidf': tfidf_recommender,
    'lsa': lsa_recommender,
    'pca': pca_recommender
}

clustering_results = perform_clustering_evaluation(recommender_instances, n_clusters=5)

best_clustering_method, clustering_scores = visualize_clustering_results(clustering_results)

print(f"\n상세 클러스터링 분석:")
for method, result in clustering_results.items():
    print(f"\n{method}:")
    print(f"  - 클러스터 균형도: {result['balance_ratio']:.3f}")
    print(f"  - 클러스터 내 거리 합: {result['inertia']:.2f}")
    
    sizes = list(result['cluster_sizes'].values())
    print(f"  - 최대 클러스터 크기: {max(sizes)}")
    print(f"  - 최소 클러스터 크기: {min(sizes)}")
    print(f"  - 클러스터 크기 편차: {np.std(sizes):.1f}")

모든 방법에 대한 클러스터링 품질 평가
K-means 클러스터링 평가 시작 (클러스터 수: 5)...
특징 행렬 수집 중...

TF-IDF 특징 공간 클러스터링...
  - 특징 행렬 크기: (16078, 1000)
  - 평가 지표 계산 중...
  클러스터링 결과:
    Silhouette Score: -0.003 (높을수록 좋음)
    Calinski-Harabasz: 151.053 (높을수록 좋음)
    Davies-Bouldin: 7.077 (낮을수록 좋음)
    클러스터 균형도: 0.863 (1에 가까울수록 균등)
    클러스터링 시간: 1.887초
    클러스터 크기: {0: 4162, 1: 1172, 2: 1876, 3: 7178, 4: 1690}
    종합 평가: 부족

LSA 특징 공간 클러스터링...
  - 특징 행렬 크기: (16078, 50)
  - 평가 지표 계산 중...
  클러스터링 결과:
    Silhouette Score: 0.058 (높을수록 좋음)
    Calinski-Harabasz: 724.184 (높을수록 좋음)
    Davies-Bouldin: 3.173 (낮을수록 좋음)
    클러스터 균형도: 0.864 (1에 가까울수록 균등)
    클러스터링 시간: 0.121초
    클러스터 크기: {0: 1862, 1: 1190, 2: 7119, 3: 4219, 4: 1688}
    종합 평가: 부족

PCA 특징 공간 클러스터링...
  - 특징 행렬 크기: (16078, 50)
  - 평가 지표 계산 중...
  클러스터링 결과:
    Silhouette Score: 0.058 (높을수록 좋음)
    Calinski-Harabasz: 746.962 (높을수록 좋음)
    Davies-Bouldin: 3.098 (낮을수록 좋음)
    클러스터 균형도: 0.859 (1에 가까울수록 균등)
    클러스터링 시간: 0.097초
    클러스터 크기: {0: 6716, 1: 1557, 2: 882,

### 06: 종합 성능 비교 분석
모든 평가 결과를 종합하여 추천 성능, 클러스터링 품질, 계산 효율성을 고려한 최적의 추천 방법론을 도출합니다. 

사용 시나리오별 최적 방법을 제안하고 실무 적용을 위한 상세한 가이드라인을 제시합니다.

* 모든 방법의 다양성, 속도, 클러스터링 품질을 종합한 비교 분석표 생성
* 지표별 가중치를 적용한 정규화 점수 계산으로 종합 성능 순위 도출
* 레이더 차트, 막대 그래프 등 다각도 시각화로 최종 비교 결과 표시
* 구현 우선순위, 기술적 고려사항, 방법별 장단점을 포함한 종합 분석 보고서 생성

In [ ]:
# ============================================ 
# Cell 1: 종합 비교 분석 함수 정의
# ============================================ 

print("\n" + "="*60)
print("6. 종합 비교 성능 지표 및 최적 방법론 도출")
print("="*60)

all_results = {
    'TF-IDF': tfidf_results,
    'LSA': lsa_results,
    'PCA': pca_results
}

def create_comprehensive_comparison():
    """
    모든 평가 결과를 종합하여 비교 분석표 생성
    
    비교 항목:
    1. 추천 다양성 (Diversity): 추천된 제품들의 중복 정도
    2. 추천 속도 (Speed): 추천 생성에 걸리는 시간
    3. 클러스터링 품질 (Clustering Quality): 특징 공간의 구조적 품질
    4. 메모리 효율성: 데이터 저장 및 처리 효율성
    5. 해석 가능성: 결과를 이해하고 설명할 수 있는 정도
    
    Returns:
    --------
    pd.DataFrame
        종합 비교 분석표
    """
    print("="*80)
    print("종 합   성 능   비 교   분 석")
    print("="*80)
    
    methods = list(all_results.keys())
    
    comparison_data = {
        'Method': methods,
        'Diversity': [all_results[m]['diversity'] for m in methods],
        'Speed_sec': [all_results[m]['avg_time'] for m in methods],
        'Clustering_Quality': [clustering_results[m]['silhouette'] for m in methods],
        'Unique_Recs': [all_results[m]['unique_recommendations'] for m in methods],
        'Matrix_Size': [str(all_results[m]['matrix_shape']) for m in methods]
    }
    
    comparison_data['Explained_Variance'] = [
        all_results['TF-IDF'].get('sparsity', 0),        
        all_results['LSA'].get('explained_variance', 0),  
        all_results['PCA'].get('explained_variance', 0)  
    ]
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print("\n핵심 성능 지표 비교:")
    print("-" * 80)
    
    for idx, row in comparison_df.iterrows():
        method = row['Method']
        print(f"\n{method}:")
        print(f"   추천 다양성: {row['Diversity']:.3f}")
        print(f"   평균 속도: {row['Speed_sec']:.4f}초")
        print(f"   클러스터링 품질: {row['Clustering_Quality']:.3f}")
        print(f"   고유 추천 수: {row['Unique_Recs']}개")
        print(f"   행렬 크기: {row['Matrix_Size']}")
    
    print(f"\n부문별 1위:")
    print(f"   최고 다양성: {methods[comparison_df['Diversity'].argmax()]} ({comparison_df['Diversity'].max():.3f})")
    print(f"   최고 속도: {methods[comparison_df['Speed_sec'].argmin()]} ({comparison_df['Speed_sec'].min():.4f}초)")
    print(f"   최고 클러스터링: {methods[comparison_df['Clustering_Quality'].argmax()]} ({comparison_df['Clustering_Quality'].max():.3f})")
    
    return comparison_df

def calculate_comprehensive_scores(comparison_df):
    """
    모든 평가 지표를 종합하여 최종 점수 계산
    
    점수 계산 방식:
    1. 각 지표를 0-1 범위로 정규화
    2. 지표별 가중치 적용
    3. 가중 평균으로 종합 점수 산출
    
    가중치 설정:
    - 추천 다양성: 30% (추천 품질의 핵심)
    - 클러스터링 품질: 30% (특징 공간 우수성)
    - 속도: 25% (실용성)
    - 안정성: 15% (기타 요소)
    
    Parameters:
    -----------
    comparison_df : pd.DataFrame
        비교 분석표
    
    Returns:
    --------
    dict
        방법별 종합 점수 및 순위
    """
    print("\n종합 점수 계산 중...")
    
    methods = comparison_df['Method'].tolist()
    
    diversity_scores = comparison_df['Diversity'].values
    norm_diversity = diversity_scores / diversity_scores.max() if diversity_scores.max() > 0 else np.zeros_like(diversity_scores)
    
    speed_scores = comparison_df['Speed_sec'].values
    inverse_speed = 1.0 / speed_scores
    norm_speed = inverse_speed / inverse_speed.max()
    
    clustering_scores = comparison_df['Clustering_Quality'].values
    norm_clustering = clustering_scores / clustering_scores.max() if clustering_scores.max() > 0 else np.zeros_like(clustering_scores)
    
    weights = {
        'diversity': 0.30,      
        'clustering': 0.30,     
        'speed': 0.25,          
        'stability': 0.15       
    }
    
    stability_scores = np.array([0.9, 0.8, 0.7])  
    
    comprehensive_scores = (
        weights['diversity'] * norm_diversity +
        weights['clustering'] * norm_clustering +
        weights['speed'] * norm_speed +
        weights['stability'] * stability_scores
    )
    
    score_results = {}
    for i, method in enumerate(methods):
        score_results[method] = {
            'comprehensive_score': comprehensive_scores[i],
            'diversity_norm': norm_diversity[i],
            'speed_norm': norm_speed[i],
            'clustering_norm': norm_clustering[i],
            'stability_score': stability_scores[i]
        }
    
    sorted_methods = sorted(methods, key=lambda x: score_results[x]['comprehensive_score'], reverse=True)
    
    print(f"\n종합 성능 순위 (상대 평가):")
    for rank, method in enumerate(sorted_methods, 1):
        score = score_results[method]['comprehensive_score']
        print(f"   {rank}위. {method:8s}: {score:.3f}점")
        print(f"        (다양성: {score_results[method]['diversity_norm']:.3f}, "
              f"속도: {score_results[method]['speed_norm']:.3f}, "
              f"클러스터링: {score_results[method]['clustering_norm']:.3f})")
    
    return score_results, sorted_methods


6. 종합 비교 성능 지표 및 최적 방법론 도출


In [29]:
# ============================================ 
# Cell 2: 종합 시각화 함수 정의
# ============================================ 

def visualize_final_comparison(comparison_df, score_results):
    """
    최종 종합 비교 결과를 다각도로 시각화
    
    시각화 구성:
    1. 핵심 지표별 레이더 차트
    2. 종합 점수 순위 막대 그래프
    3. 방법별 장단점 매트릭스
    4. 사용 시나리오별 추천 가이드
    
    Parameters:
    -----------
    comparison_df : pd.DataFrame
        비교 분석표
    score_results : dict
        종합 점수 결과
    """
    print("\n최종 비교 결과 시각화...")
    
    methods = comparison_df['Method'].tolist()
    
    # 테이블 형식으로 성능 비교 출력
    print("\n" + "="*100)
    print("성능 비교 매트릭스")
    print("="*100)
    
    # 다양성 비교
    print("\n추천 다양성 비교:")
    print(f"{'방법':^10} | {'다양성':^10} | {'평가':^20}")
    print("-" * 45)
    diversity_scores = comparison_df['Diversity'].values
    max_diversity_idx = np.argmax(diversity_scores)
    for i, method in enumerate(methods):
        star = "★" if i == max_diversity_idx else ""
        print(f"{method:^10} | {diversity_scores[i]:^10.3f} | {'우수' if diversity_scores[i] > 0.7 else '보통':^20} {star}")
    
    # 속도 비교
    print("\n추천 생성 속도:")
    print(f"{'방법':^10} | {'속도(초)':^10} | {'평가':^20}")
    print("-" * 45)
    speed_scores = comparison_df['Speed_sec'].values
    min_speed_idx = np.argmin(speed_scores)
    for i, method in enumerate(methods):
        star = "★" if i == min_speed_idx else ""
        print(f"{method:^10} | {speed_scores[i]:^10.4f} | {'빠름' if speed_scores[i] < 0.01 else '보통':^20} {star}")
    
    # 클러스터링 품질 비교
    print("\n클러스터링 품질:")
    print(f"{'방법':^10} | {'Silhouette':^12} | {'평가':^20}")
    print("-" * 45)
    clustering_scores = comparison_df['Clustering_Quality'].values
    max_clustering_idx = np.argmax(clustering_scores)
    for i, method in enumerate(methods):
        star = "★" if i == max_clustering_idx else ""
        quality = '매우 우수' if clustering_scores[i] > 0.7 else ('우수' if clustering_scores[i] > 0.5 else '보통')
        print(f"{method:^10} | {clustering_scores[i]:^12.3f} | {quality:^20} {star}")
    
    # 종합 점수
    print("\n종합 성능 점수:")
    print(f"{'방법':^10} | {'종합점수':^10} | {'순위':^10}")
    print("-" * 35)
    comprehensive_scores = [score_results[m]['comprehensive_score'] for m in methods]
    sorted_indices = np.argsort(comprehensive_scores)[::-1]
    for rank, idx in enumerate(sorted_indices, 1):
        method = methods[idx]
        score = comprehensive_scores[idx]
        trophy = "🏆" if rank == 1 else ""
        print(f"{method:^10} | {score:^10.3f} | {rank:^10} {trophy}")
    
    # 방법별 특징
    print("\n" + "="*100)
    print("방법별 핵심 특징")
    print("="*100)
    
    feature_text = """
TF-IDF:
• 빠른 속도, 해석 용이
• 원본 특징 보존
• 희소성 활용

LSA:
• 의미적 관계 포착
• 노이즈 감소 효과
• 차원 축소 최적화

PCA:
• 분산 보존 극대화
• 통계적 해석 가능
• 주성분 분석
"""
    print(feature_text)
    
    # 사용 시나리오별 권장사항
    print("\n" + "="*100)
    print("사용 시나리오별 권장사항")
    print("="*100)
    
    scenario_text = """
실시간 서비스:
→ TF-IDF (속도 우선)

품질 중심 서비스:
→ LSA (의미적 유사성)

분석 목적:
→ PCA (통계적 해석)

하이브리드:
→ 상황별 조합 활용
"""
    print(scenario_text)
    
    best_method = methods[np.argmax(comprehensive_scores)]
    best_score = max(comprehensive_scores)
    
    # 최종 권장사항
    print("\n" + "="*100)
    print("최종 분석 결과 및 권장사항")
    print("="*100)
    print(f"\n종합 1위: {best_method} (점수: {best_score:.3f})")
    print("\n부문별 우승:")
    print(f"• 다양성: {methods[np.argmax(diversity_scores)]}")
    print(f"• 속도: {methods[np.argmin(speed_scores)]}")
    print(f"• 품질: {methods[np.argmax(clustering_scores)]}")
    print("\n실무 권장:")
    print(f"• 시작 단계: TF-IDF")
    print(f"• 성능 중시: {best_method}")
    
    return best_method

In [ ]:
# ============================================ 
# Cell 3: 최종 분석 보고서 함수 정의
# ============================================ 

def generate_final_insights_report(score_results, sorted_methods, clustering_results):
    """
    모든 평가 결과를 종합한 최종 분석 보고서 생성
    
    보고서 구성:
    1. 실험 개요 및 데이터 요약
    2. 방법별 상세 성능 분석
    3. 주요 발견사항 및 인사이트
    4. 실무 적용 가이드라인
    5. 한계점 및 향후 개선방안
    
    Parameters:
    -----------
    score_results : dict
        종합 점수 결과
    sorted_methods : list
        성능 순위별 정렬된 방법 목록
    clustering_results : dict
        클러스터링 평가 결과
    """
    print("\n" + "="*100)
    print("최 종   종 합   분 석   보 고 서")
    print("="*100)
    
    print(f"\n1. 실험 개요:")
    print(f"   데이터 규모:")
    print(f"      - 총 리뷰 수: {len(df):,}개")
    print(f"      - 고유 제품 수: {len(product_texts):,}개")
    print(f"      - 평균 제품당 리뷰: {len(df)/len(product_texts):.1f}개")
    print(f"   평가 방법: TF-IDF, LSA, PCA")
    print(f"   평가 지표: 다양성, 속도, 클러스터링 품질")
    
    print(f"\n2. 방법별 상세 성능 분석:")
    
    for rank, method in enumerate(sorted_methods, 1):
        result = all_results[method]
        cluster_result = clustering_results[method]
        score = score_results[method]
        
        print(f"\n   {rank}위. {method} (종합 상대 점수: {score['comprehensive_score']:.3f})")
        print(f"      추천 성능:")
        print(f"         - 다양성: {result['diversity']:.3f}")
        print(f"         - 속도: {result['avg_time']:.4f}초")
        print(f"         - 고유 추천: {result['unique_recommendations']}개")
        
        print(f"      클러스터링 성능:")
        print(f"         - Silhouette: {cluster_result['silhouette']:.3f}")
        print(f"         - 균형도: {cluster_result['balance_ratio']:.3f}")
        print(f"         - 클러스터링 시간: {cluster_result['clustering_time']:.3f}초")
        
        print(f"      효율성:")
        print(f"         - 행렬 크기: {result['matrix_shape']}")
        
        if method == 'TF-IDF' and 'sparsity' in result:
            print(f"         - 희소성: {result['sparsity']:.3f}")
        elif method in ['LSA', 'PCA'] and 'explained_variance' in result:
            print(f"         - 설명 분산: {result['explained_variance']:.3f}")
    
    best_overall = sorted_methods[0]
    fastest_method = min(all_results.keys(), key=lambda x: all_results[x]['avg_time'])
    most_diverse = max(all_results.keys(), key=lambda x: all_results[x]['diversity'])
    best_clustering = max(clustering_results.keys(), key=lambda x: clustering_results[x]['silhouette'])
    
    print(f"\n3. 주요 발견사항 및 인사이트:")
    print(f"   핵심 발견:")
    print(f"      종합 최우수: {best_overall}")
    print(f"         → 균형 잡힌 성능으로 대부분 상황에 적합")
    print(f"      최고 속도: {fastest_method} ({all_results[fastest_method]['avg_time']:.4f}초)")
    print(f"         → 실시간 서비스에 최적")
    print(f"      최고 다양성: {most_diverse} ({all_results[most_diverse]['diversity']:.3f})")
    print(f"         → 추천 품질이 중요한 서비스에 적합")
    print(f"      최고 구조: {best_clustering} (Silhouette: {clustering_results[best_clustering]['silhouette']:.3f})")
    print(f"         → 데이터 구조화가 중요한 분석에 적합")
    
    scores = [score_results[m]['comprehensive_score'] for m in sorted_methods]
    max_gap = scores[0] - scores[-1]
    print(f"      성능 격차: {max_gap:.3f} (1위와 3위 차이)")
    
    if max_gap < 0.1:
        gap_analysis = "방법 간 성능 차이가 작아 상황에 따른 선택이 중요"
    elif max_gap < 0.2:
        gap_analysis = "적정한 성능 차이로 명확한 우열 구분 가능"
    else:
        gap_analysis = "큰 성능 차이로 최적 방법 선택이 매우 중요"
    
    print(f"         → {gap_analysis}")

    print("\n4. 방법별 장단점:")
    
    print("\n   TF-IDF:")
    print("   장점: 해석 용이, 빠른 속도, 원본 특징 보존")
    print("   단점: 의미적 관계 포착 한계, 희소성 문제")
    
    print("\n   LSA:")
    print("   장점: 의미적 유사성 포착, 노이즈 감소, 차원 축소")
    print("   단점: 음수 값 가능, 해석의 어려움")
    
    print("\n   PCA:")
    print("   장점: 분산 최대화, 차원 축소 효과적")
    print("   단점: 텍스트 데이터에 덜 적합, 의미 해석 어려움")
    
    # 3. 사용 시나리오별 권장사항
    print("\n5. 사용 시나리오별 권장사항:")
    print("\n   - 실시간 추천이 중요한 경우: TF-IDF")
    print("   - 의미적 유사성이 중요한 경우: LSA")
    print("   - 차원 축소가 주목적인 경우: PCA 또는 LSA")
   

print("최종 종합 분석 시작...")

comparison_df = create_comprehensive_comparison()

score_results, sorted_methods = calculate_comprehensive_scores(comparison_df)

best_method = visualize_final_comparison(comparison_df, score_results)

generate_final_insights_report(score_results, sorted_methods, clustering_results)

최종 종합 분석 시작...
종 합   성 능   비 교   분 석

핵심 성능 지표 비교:
--------------------------------------------------------------------------------

TF-IDF:
   추천 다양성: 0.990
   평균 속도: 0.0187초
   클러스터링 품질: -0.003
   고유 추천 수: 99개
   행렬 크기: (16078, 1000)

LSA:
   추천 다양성: 0.990
   평균 속도: 0.0070초
   클러스터링 품질: 0.058
   고유 추천 수: 99개
   행렬 크기: (16078, 50)

PCA:
   추천 다양성: 0.990
   평균 속도: 0.0075초
   클러스터링 품질: 0.058
   고유 추천 수: 99개
   행렬 크기: (16078, 50)

부문별 1위:
   최고 다양성: TF-IDF (0.990)
   최고 속도: LSA (0.0070초)
   최고 클러스터링: PCA (0.058)

종합 점수 계산 중...

종합 성능 순위:
   1위. LSA     : 0.969점
        (다양성: 1.000, 속도: 1.000, 클러스터링: 0.997)
   2위. PCA     : 0.940점
        (다양성: 1.000, 속도: 0.939, 클러스터링: 1.000)
   3위. TF-IDF  : 0.513점
        (다양성: 1.000, 속도: 0.376, 클러스터링: -0.053)

최종 비교 결과 시각화...

성능 비교 매트릭스

추천 다양성 비교:
    방법     |    다양성     |          평가         
---------------------------------------------
  TF-IDF   |   0.990    |          우수          ★
   LSA     |   0.990    |          우수          
   PCA     |   0